# Route C — BERTweet (revised)

Drop-in replacement for `vu-dsp2.ipynb`. Implements the five review points plus
several runtime reductions, and produces a **deployable `.model`** in the
case-manual format.

**What changed**
1. **BERTweet** (`vinai/bertweet-base`) instead of `bert-base-uncased`, with the
   tokenizer's own `@USER`/`HTTPURL`/emoji normalisation (no MNTN/URL skew).
2. **Canonical `data_split.csv`** — train/val/test come from the *same* split as
   Routes A/B; the final number is reported on the **test** partition.
3. **Deployable `.model`** — a self-contained, picklable
   `{"vectorizer", "classifier"}` dict (classes live in `sentiment_deploy.py`,
   not `__main__`); labels map back to the API space **{-1, 0, 1}**.
4. **`max_length = 128`** + **dynamic padding** (no padding to a global max).
5. **Class weighting** — inverse-frequency weights that **upweight Neutral**
   (the minority + hardest class) via a custom `WeightedTrainer`.

**Extra runtime reductions** (mirroring the A/B notebooks): mixed precision
(`fp16`), length-bucketed batching (`group_by_length`), parallel tokenisation,
early stopping, larger eval batch, optional bottom-layer freezing, and
**grid search on a stratified subsample** with the final model retrained on the
full training split.

**On grid search + 5-fold CV.** For the classical Routes A/B, `GridSearchCV(cv=5)`
is cheap. For a transformer it is *not*: each fold is a full fine-tune of ~135M
parameters, so 5-fold × a grid would mean dozens of full training runs. The
applicable form here is a **validation-based grid search** (the standard for
transformer fine-tuning), implemented below. A real `StratifiedKFold` path is
included and can be switched on with `N_FOLDS > 1`, but be aware it multiplies
runtime by `N_FOLDS`.

In [1]:
# ============================ CONFIG ============================
from pathlib import Path

# --- data ---
SPLIT_CSV   = "data_split.csv"   # canonical split shared with Routes A/B
TEXT_COL    = None    # None -> auto-detect among common names
LABEL_COL   = None    # None -> auto-detect; values may be {-1,0,1}, {0,1,2} or stars {1..5}
SPLIT_COL   = None    # None -> auto-detect ('split'/'set'/'partition')
TRAIN_KEYS  = {"train", "training"}
VAL_KEYS    = {"val", "valid", "validation", "dev"}
TEST_KEYS   = {"test", "testing", "holdout"}

# --- model ---
MODEL_NAME  = "vinai/bertweet-base"
MAX_LEN     = 128
NUM_LABELS  = 3

# --- cross-architecture ensemble ---
# Both heads are neg/neu/pos == internal {0,1,2}; twitter-roberta is
# sentiment-pretrained so its head warm-starts and it errs differently from
# BERTweet. The grid below picks (lr, weighted) on BERTweet; that winning
# config is reused to fine-tune BOTH members for the final ensemble.
MODELS = [
    "vinai/bertweet-base",
    "cardiffnlp/twitter-roberta-base-sentiment-latest",
]

# --- per-class threshold tuning ---
# Offsets fit on VALIDATION logits only, then applied unchanged to TEST.
THRESH_GRID = None   # None -> np.linspace(-3, 3, 61) inside the tuner

# --- training ---
SEED          = 42
EPOCHS        = 5            # was 4 — one more epoch; early stopping + best-model-at-end guard overfit
TRAIN_BS = 16    # was 32
EVAL_BS  = 32    # was 64
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.06
EARLY_STOP    = 2            # was 1 — more patience so a transient dip does not stop training early
FREEZE_BOTTOM = 0            # freeze embeddings + first N encoder layers (0 = full FT)

# --- grid search (validation-based) ---
GRID_LR        = [1e-5, 2e-5, 3e-5]   # was [2e-5,3e-5] — wider LR search now that F1 > speed
GRID_WEIGHTED  = [True, False]   # was [True, False] — halves the grid (2 runs not 4)
GRID_SUBSAMPLE = 20000          # was 8000 — larger search pool gives a less noisy config choice
GRID_EPOCHS    = 2               # was 1 — one epoch is too noisy to rank configs reliably
N_FOLDS        = 1               # keep at 1 — KFold multiplies time by k, don't enable it

# --- runtime ---
FP16            = True
GROUP_BY_LENGTH = True
TOKENIZE_PROC   = 4          # parallel tokenisation workers

# --- output ---
OUT_DIR     = Path("route_c_out");  OUT_DIR.mkdir(exist_ok=True)
MODEL_FILE  = "route_c_bertweet.model"
ROUTE_A_F1  = 0.68           # baseline for context

import random, numpy as np
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass
print("Config loaded.")

Config loaded.


🛠 **what changed (this version):** three F1 levers stacked on top of the previous file.
1. **5 epochs** instead of 4 (early stopping + `load_best_model_at_end` keep the best checkpoint regardless).
2. **Cross-architecture ensemble** — the winning grid config is used to fine-tune **both** `vinai/bertweet-base` and `cardiffnlp/twitter-roberta-base-sentiment-latest`; their logits are **log-averaged** (geometric mean of probabilities). Each model is tokenised with its *own* tokenizer.
3. **Per-class threshold tuning** — additive per-class offsets are fit on the **validation** ensemble logits to maximise macro-F1, then applied unchanged to **test** (no leakage). This is the cheap fix for Neutral being under-predicted by argmax.

The grid search still runs on BERTweet only (searching `lr × weighted`) to keep runtime sane. Expect roughly **2× the training time** of the single-model version, since two full fine-tunes now happen in the final stage.

In [2]:
# ============================================================
# Route C (BERTweet) — Colab environment check
# Run this FIRST. It verifies GPU, libraries, model access,
# and data before you attempt the full training run.
# ============================================================
import sys, importlib, traceback

problems = []

def ok(msg):   print("  [OK]  " + msg)
def warn(msg): print("  [!!]  " + msg);
def bad(msg):  print("  [XX]  " + msg); problems.append(msg)

print("="*60)
print("1. GPU / CUDA")
print("="*60)
try:
    import torch
    if torch.cuda.is_available():
        ok(f"CUDA available — {torch.cuda.get_device_name(0)}")
        ok(f"torch {torch.__version__}, CUDA build {torch.version.cuda}")
        free, total = torch.cuda.mem_get_info()
        ok(f"GPU memory: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
    else:
        bad("CUDA NOT available — you are on CPU. "
            "Runtime → Change runtime type → T4 GPU, then restart and rerun.")
except Exception as e:
    bad(f"torch import failed: {e}")

print("="*60)
print("2. Core libraries")
print("="*60)
need = ["numpy", "pandas", "transformers", "tokenizers",
        "sklearn", "bs4"]   # bs4 = BeautifulSoup (HTML cleaning)
for name in need:
    try:
        m = importlib.import_module(name)
        ver = getattr(m, "__version__", "?")
        ok(f"{name} {ver}  ({m.__file__})")
    except Exception as e:
        bad(f"{name} missing/broken: {e}")

# emoji is optional — only needed if you demojize yourself (you don't have to)
try:
    import emoji
    ok(f"emoji {emoji.__version__} (optional)")
except Exception:
    warn("emoji not installed (optional — BERTweet's normalizer handles emojis)")

print("="*60)
print("3. numpy / pandas ABI consistency")
print("="*60)
try:
    import numpy as np, pandas as pd
    # the exact operation that failed on the VU hub:
    _ = pd.DataFrame({"a": np.array([1, 2, 3])})
    ok("pandas can build a DataFrame from a numpy array (no ABI mismatch)")
    npdir = np.__file__.split("site-packages")[0]
    pddir = pd.__file__.split("site-packages")[0]
    if npdir == pddir:
        ok("numpy and pandas live in the same environment")
    else:
        bad(f"numpy ({npdir}) and pandas ({pddir}) are in DIFFERENT trees — ABI risk")
except Exception as e:
    bad(f"numpy/pandas ABI check failed: {e}")

print("="*60)
print("4. BERTweet model + tokenizer download")
print("="*60)
MODEL_NAME = "vinai/bertweet-base"
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, normalization=True, use_fast=False)
    ok(f"tokenizer loaded ({MODEL_NAME})")
    # confirm the tweet-normalizer actually fires (the BERTweet advantage)
    sample = "@someone check http://x.com this is GREAT 😡"
    enc = tok.tokenize(sample)
    norm_hit = any(t in {"@USER", "HTTPURL"} for t in enc) or "@@" in " ".join(enc)
    if norm_hit or "HTTPURL" in tok.decode(tok.encode(sample)):
        ok("normalization=True is active (mentions/URLs/emoji handled by tokenizer)")
    else:
        warn("could not confirm normalizer fired — inspect tok.tokenize(sample) manually")
    m = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    ok("model loaded with a fresh 3-class head "
       "(MISSING classifier.* keys at load time is NORMAL — those are the new head)")
    del m
except Exception as e:
    bad(f"model/tokenizer load failed: {e}")
    traceback.print_exc()

print("="*60)
print("5. Data file")
print("="*60)
# >>> EDIT this to wherever you put data_split.csv in Colab <
SPLIT_CSV = "data_split.csv"
import os
try:
    if not os.path.exists(SPLIT_CSV):
        bad(f"{SPLIT_CSV} not found. Upload it (left Files panel) or mount Drive, "
            f"then set SPLIT_CSV to the right path.")
    else:
        import pandas as pd
        df = pd.read_csv(SPLIT_CSV)
        ok(f"loaded {SPLIT_CSV} — shape {df.shape}")
        ok(f"columns: {list(df.columns)}")
        # quick check for the split + label columns the notebook expects
        cols = {c.lower() for c in df.columns}
        if cols & {"split", "set", "partition", "subset", "fold"}:
            ok("a split column is present (test set will be comparable to Routes A/B)")
        else:
            warn("no split column found — Route C cell 8 will make a FRESH split "
                 "(NOT comparable to A/B). Check this before trusting the F1.")
        if cols & {"text", "review", "review_cleaned", "document", "content", "sentence"}:
            ok("a text column is present")
        else:
            warn("no obvious text column — set TEXT_COL manually in the config")
except Exception as e:
    bad(f"data check failed: {e}")

print("="*60)
print("SUMMARY")
print("="*60)
if not problems:
    print("  All critical checks passed — you're clear to run Route C.")
else:
    print(f"  {len(problems)} blocking issue(s) — fix these before training:")
    for p in problems:
        print("   - " + p)
print("="*60)

1. GPU / CUDA
  [OK]  CUDA available — NVIDIA GeForce RTX 5070 Ti
  [OK]  torch 2.11.0+cu128, CUDA build 12.8
  [OK]  GPU memory: 15.8 GB free / 17.1 GB total
2. Core libraries
  [OK]  numpy 2.4.6  (C:\Users\Lockie\.venv\Lib\site-packages\numpy\__init__.py)
  [OK]  pandas 3.0.3  (C:\Users\Lockie\.venv\Lib\site-packages\pandas\__init__.py)
  [OK]  transformers 5.12.0  (C:\Users\Lockie\.venv\Lib\site-packages\transformers\__init__.py)
  [OK]  tokenizers 0.22.2  (C:\Users\Lockie\.venv\Lib\site-packages\tokenizers\__init__.py)
  [OK]  sklearn 1.9.0  (C:\Users\Lockie\.venv\Lib\site-packages\sklearn\__init__.py)
  [OK]  bs4 4.15.0  (C:\Users\Lockie\.venv\Lib\site-packages\bs4\__init__.py)
  [OK]  emoji 0.6.0 (optional)
3. numpy / pandas ABI consistency
  [OK]  pandas can build a DataFrame from a numpy array (no ABI mismatch)
  [OK]  numpy and pandas live in the same environment
4. BERTweet model + tokenizer download


  [OK]  tokenizer loaded (vinai/bertweet-base)
  [OK]  normalization=True is active (mentions/URLs/emoji handled by tokenizer)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [OK]  model loaded with a fresh 3-class head (MISSING classifier.* keys at load time is NORMAL — those are the new head)
5. Data file
  [OK]  loaded data_split.csv — shape (255083, 6)
  [OK]  columns: ['Unnamed: 0', 'Text', 'clean_eda', 'sentiment', 'Type', 'split']
  [OK]  a split column is present (test set will be comparable to Routes A/B)
  [OK]  a text column is present
SUMMARY
  All critical checks passed — you're clear to run Route C.


## 1. Write the picklable deployment module

Writing the wrapper classes to a real `.py` file (imported below) is what makes the pickle load in the API's separate process — objects defined in a notebook live in `__main__` and fail to unpickle elsewhere. Ship this file next to `app.py`.

In [3]:
%%writefile sentiment_deploy.py
"""
sentiment_deploy.py
===================

Self-contained, picklable deployment wrapper for the Route C (BERTweet)
sentiment classifier, compatible with the case-manual API template.

The API loads a single ``*.model`` pickle and expects a dict::

    {"vectorizer": <obj with .transform(list[str])>,
     "classifier": <obj with .predict(X)>}

A HuggingFace transformer does not fit that interface, so this module
provides two small adapters:

* ``BertweetVectorizer`` -- a pass-through "vectorizer". It applies the SAME
  light cleaning used at training time and returns the (cleaned) strings.
  ``fit`` / ``fit_transform`` are no-ops, so the wrapper is safe even if the
  API template erroneously calls ``fit_transform`` at inference time.

* ``BertweetClassifier`` -- holds the fine-tuned weights, config and tokenizer
  files *inside the pickle* (no external paths, no dependence on a checkpoint
  directory). It rebuilds the model lazily on first use and maps the internal
  class indices {0,1,2} back to the API label space {-1, 0, 1}.

IMPORTANT (pickle/__main__ caveat from earlier): because the API loads the
pickle in a *separate process*, the classes referenced by the pickle must be
importable there. Defining them in THIS module (not in a notebook's __main__)
is what makes the round-trip work. Ship ``sentiment_deploy.py`` alongside the
API ``app.py``.
"""

from __future__ import annotations

import io
import os
import tempfile
from typing import List, Sequence

# Heavy deps (torch / transformers / bs4) are imported lazily inside methods
# so this module can be imported in lightweight contexts and unit-tested.

# Internal index -> API label.  Training uses 0=Negative, 1=Neutral, 2=Positive.
# The API/case-manual label space is -1=Negative, 0=Neutral, 1=Positive.
INDEX_TO_API_LABEL = {0: -1, 1: 0, 2: 1}


def normalize_text(x) -> str:
    """Light, rule-based cleaning applied identically at train and serve time.

    Only does what BERTweet's own tokenizer normalisation does NOT do: strip
    HTML (reviews contain markup) and collapse whitespace. Mention/URL/emoji
    handling is delegated to the tokenizer (``normalization=True``) so that
    train and serve stay perfectly consistent and there is no MNTN/URL skew.
    """
    if x is None:
        return ""
    x = str(x)
    if "<" in x and ">" in x:  # only pay BeautifulSoup cost when markup is likely
        try:
            from bs4 import BeautifulSoup
            x = BeautifulSoup(x, "html.parser").get_text(separator=" ")
        except Exception:
            pass
    x = " ".join(x.split())  # collapse all whitespace runs
    return x


class BertweetVectorizer:
    """Pass-through 'vectorizer' for API compatibility.

    Tokenisation happens inside the classifier, so ``transform`` just returns
    the cleaned strings. ``fit``/``fit_transform`` are no-ops -- importantly,
    ``fit_transform`` does NOT re-fit anything, so the buggy template call
    ``vectorizer.fit_transform(text)`` at inference behaves like ``transform``.
    """

    def fit(self, X=None, y=None):
        return self

    def transform(self, X: Sequence[str]) -> List[str]:
        if isinstance(X, str):
            X = [X]
        return [normalize_text(t) for t in X]

    def fit_transform(self, X: Sequence[str], y=None) -> List[str]:
        return self.transform(X)


class BertweetClassifier:
    """Self-contained, picklable BERTweet sequence classifier.

    Parameters
    ----------
    model : transformers PreTrainedModel (fine-tuned)
    tokenizer : transformers PreTrainedTokenizer
    max_length : int
    batch_size : int
        Inference batch size.
    """

    def __init__(self, model=None, tokenizer=None, max_length: int = 128,
                 batch_size: int = 64):
        self.max_length = int(max_length)
        self.batch_size = int(batch_size)
        self.index_to_api = dict(INDEX_TO_API_LABEL)

        # Serialised payload (populated from the live objects). Kept so the
        # pickle is fully self-contained.
        self._config = None
        self._state_dict = None       # dict[str, torch.Tensor] on CPU
        self._tokenizer_files = None  # dict[str, bytes]

        if model is not None and tokenizer is not None:
            self._capture(model, tokenizer)

        # Live objects (rebuilt lazily; never pickled).
        self._model = None
        self._tok = None

    # ---- serialisation helpers -------------------------------------------
    def _capture(self, model, tokenizer):
        """Snapshot weights/config/tokenizer into picklable payload."""
        self._config = model.config
        self._state_dict = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        with tempfile.TemporaryDirectory() as d:
            tokenizer.save_pretrained(d)
            files = {}
            for name in os.listdir(d):
                path = os.path.join(d, name)
                if os.path.isfile(path):
                    with open(path, "rb") as fh:
                        files[name] = fh.read()
            self._tokenizer_files = files

    def __getstate__(self):
        # Exclude live (non-portable) objects from the pickle.
        return {
            "max_length": self.max_length,
            "batch_size": self.batch_size,
            "index_to_api": self.index_to_api,
            "_config": self._config,
            "_state_dict": self._state_dict,
            "_tokenizer_files": self._tokenizer_files,
        }

    def __setstate__(self, state):
        self.__dict__.update(state)
        self._model = None
        self._tok = None

    # ---- lazy rebuild -----------------------------------------------------
    def _build_model(self, config, state_dict):
        """Rebuild the HF model from config + state_dict (no hub download)."""
        from transformers import AutoModelForSequenceClassification
        model = AutoModelForSequenceClassification.from_config(config)
        model.load_state_dict(state_dict)
        return model

    def _ensure(self):
        if self._model is not None:
            return
        import torch
        from transformers import AutoTokenizer

        # tokenizer
        self._tokdir = tempfile.mkdtemp(prefix="bertweet_tok_")
        for name, data in self._tokenizer_files.items():
            with open(os.path.join(self._tokdir, name), "wb") as fh:
                fh.write(data)
        self._tok = AutoTokenizer.from_pretrained(
            self._tokdir, normalization=True, use_fast=False)

        # model
        model = self._build_model(self._config, self._state_dict)
        self._device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(self._device)
        model.eval()
        self._model = model

    # ---- inference --------------------------------------------------------
    def predict(self, X: Sequence[str]):
        """Return a list of API labels in {-1, 0, 1} for the input texts.

        Accepts either raw strings or strings already passed through the
        vectorizer (cleaning is idempotent, so both work).
        """
        import numpy as np
        import torch

        if isinstance(X, str):
            X = [X]
        texts = [normalize_text(t) for t in X]
        self._ensure()

        preds = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i + self.batch_size]
            enc = self._tok(batch, max_length=self.max_length, truncation=True,
                            padding=True, return_tensors="pt")
            enc = {k: v.to(self._device) for k, v in enc.items()}
            with torch.no_grad():
                logits = self._model(**enc).logits
            idx = logits.argmax(dim=1).cpu().numpy()
            preds.extend(int(self.index_to_api[int(j)]) for j in idx)
        return preds

    # convenience
    def predict_proba(self, X: Sequence[str]):
        import torch
        import torch.nn.functional as F
        if isinstance(X, str):
            X = [X]
        texts = [normalize_text(t) for t in X]
        self._ensure()
        out = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i + self.batch_size]
            enc = self._tok(batch, max_length=self.max_length, truncation=True,
                            padding=True, return_tensors="pt")
            enc = {k: v.to(self._device) for k, v in enc.items()}
            with torch.no_grad():
                logits = self._model(**enc).logits
            out.append(F.softmax(logits, dim=1).cpu().numpy())
        return out and __import__("numpy").vstack(out)

Overwriting sentiment_deploy.py


In [4]:
import importlib, sentiment_deploy
importlib.reload(sentiment_deploy)
from sentiment_deploy import (
    normalize_text, BertweetVectorizer, BertweetClassifier, INDEX_TO_API_LABEL,
)
print("Imported wrapper. internal->API label map:", INDEX_TO_API_LABEL)

Imported wrapper. internal->API label map: {0: -1, 1: 0, 2: 1}


## 2. Helpers — label normalisation, column detection, class weights, metric

In [5]:
import pandas as pd, numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, classification_report

def _find_col(cols, candidates):
    low = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand in low:
            return low[cand]
    return None

def normalize_labels_to_3(series):
    # Map any of {-1,0,1} / {0,1,2} / star ratings {1..5} -> {0,1,2}
    # where 0=Negative, 1=Neutral, 2=Positive.
    s = pd.to_numeric(series, errors="coerce")
    u = set(int(v) for v in s.dropna().unique())
    if u <= {-1, 0, 1}:
        return s.map({-1: 0, 0: 1, 1: 2}).astype("int64")
    if u <= {0, 1, 2}:
        return s.astype("int64")
    if u <= {1, 2, 3, 4, 5}:
        return s.map({1: 0, 2: 0, 3: 1, 4: 2, 5: 2}).astype("int64")
    raise ValueError(f"Unrecognised label set: {u}")

def class_weights(y):
    w = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=np.asarray(y))
    return w.astype("float32")   # neutral (minority) gets the largest weight

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"f1_macro": f1_score(labels, preds, average="macro"),
            "accuracy": accuracy_score(labels, preds)}
print("Helpers ready.")

Helpers ready.


## 3. Load the canonical split

Reads `data_split.csv` (the same artifact Routes A/B use), applies the light rule-based cleaning, and slices train/val/test. If the file has no split column, it falls back to a stratified split with a clear warning.

In [6]:
import gdown
gdown.download(
    id="13mhU5Ivjn3P73-UOcXKt17EFe1N76Tys",
    output="data_split.csv",
    quiet=False,
)

Downloading...
From: https://drive.google.com/uc?id=13mhU5Ivjn3P73-UOcXKt17EFe1N76Tys
To: C:\Users\Lockie\data_split.csv
100%|██████████| 90.9M/90.9M [00:02<00:00, 37.0MB/s]


'data_split.csv'

In [7]:
raw = pd.read_csv(SPLIT_CSV)
print("Loaded", SPLIT_CSV, "shape", raw.shape, "| columns:", list(raw.columns))

text_col  = TEXT_COL  or _find_col(raw.columns, ["text","review_cleaned","review","document","content","sentence"])
label_col = LABEL_COL or _find_col(raw.columns, ["sentiment_encoded","label","sentiment","target","y","numericvalue"])
split_col = SPLIT_COL or _find_col(raw.columns, ["split","set","partition","subset","fold"])
assert text_col and label_col, f"Could not find text/label columns in {list(raw.columns)}"
print(f"Using text='{text_col}', label='{label_col}', split='{split_col}'")

# Redefine normalize_labels_to_3 to remove the .astype("int64") call
# within the function itself, allowing NaNs to persist until after dropna.
def normalize_labels_to_3(series):
    s = pd.to_numeric(series, errors="coerce")
    # Ensure unique labels are checked on non-NaN values to avoid type issues
    valid_s = s.dropna()
    if valid_s.empty:
        # If all values are NaN, we can't determine the label set
        # Returning s as is (full of NaNs) will be handled by subsequent dropna
        return s

    u = set(int(v) for v in valid_s.unique())

    if u <= {-1, 0, 1}:
        return s.map({-1: 0, 0: 1, 1: 2})
    if u <= {0, 1, 2}:
        return s
    if u <= {1, 2, 3, 4, 5}:
        return s.map({1: 0, 2: 0, 3: 1, 4: 2, 5: 2})
    raise ValueError(f"Unrecognised label set: {u}")

raw["_text"]  = raw[text_col].astype(str).map(normalize_text)
raw["labels"] = normalize_labels_to_3(raw[label_col])
raw = raw.dropna(subset=["labels"]).reset_index(drop=True)
raw["labels"] = raw["labels"].astype("int64")

if split_col is not None:
    s = raw[split_col].astype(str).str.lower().str.strip()
    train_df = raw[s.isin(TRAIN_KEYS)].copy()
    val_df   = raw[s.isin(VAL_KEYS)].copy()
    test_df  = raw[s.isin(TEST_KEYS)].copy()
    if len(val_df) == 0:               # some splits only have train/test
        from sklearn.model_selection import train_test_split
        train_df, val_df = train_test_split(train_df, test_size=0.1,
            stratify=train_df["labels"], random_state=SEED)
        print("No val partition found -> carved 10% val out of train.")
else:
    print("WARNING: no split column found -> making a fresh stratified 80/10/10 split. "
          "Results will NOT be comparable to Routes A/B unless this matches their split.")
    from sklearn.model_selection import train_test_split
    train_df, tmp = train_test_split(raw, test_size=0.2, stratify=raw["labels"], random_state=SEED)
    val_df, test_df = train_test_split(tmp, test_size=0.5, stratify=tmp["labels"], random_state=SEED)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    dist = d["labels"].value_counts().sort_index().to_dict()
    print(f"{name:5s}: {len(d):>7d}  dist(0/1/2)={dist}")

Loaded data_split.csv shape (255083, 6) | columns: ['Unnamed: 0', 'Text', 'clean_eda', 'sentiment', 'Type', 'split']
Using text='Text', label='sentiment', split='split'
train:  178557  dist(0/1/2)={0: 62851, 1: 44461, 2: 71245}
val  :   38263  dist(0/1/2)={0: 13468, 1: 9528, 2: 15267}
test :   38263  dist(0/1/2)={0: 13468, 1: 9528, 2: 15267}


## 4. Tokeniser + datasets (dynamic padding)

`normalization=True` lets BERTweet handle mentions/URLs/emojis. We tokenise once (parallelised) and pad **per batch** via `DataCollatorWithPadding` — no padding to a global 512 max.

In [8]:
from transformers import AutoTokenizer, DataCollatorWithPadding
import torch
from torch.utils.data import Dataset

def make_tokenizer(model_name):
    """BERTweet needs its slow tokenizer + tweet normalisation; RoBERTa-family
    models (e.g. twitter-roberta) use their own fast BPE tokenizer."""
    if "bertweet" in model_name.lower():
        return AutoTokenizer.from_pretrained(model_name, normalization=True, use_fast=False)
    return AutoTokenizer.from_pretrained(model_name)

def cardiff_preprocess(text):
    """twitter-roberta was trained with mentions -> '@user' and links -> 'http'.
    BERTweet's tokenizer does its own @USER/HTTPURL normalisation, so this is
    applied ONLY to the roberta member to feed each model text in its native form."""
    out = []
    for tok in str(text).split(" "):
        if tok.startswith("@") and len(tok) > 1:
            tok = "@user"
        elif tok.startswith("http"):
            tok = "http"
        out.append(tok)
    return " ".join(out)

def _prep_texts(texts, model_name):
    if "bertweet" in model_name.lower():
        return list(texts)                       # tokenizer normalises internally
    return [cardiff_preprocess(t) for t in texts]

class TokenizedDataset(Dataset):
    """Plain torch Dataset of pre-tokenized (unpadded) examples.

    Avoids the `datasets`/`pyarrow` dependency. Padding is done per batch by
    DataCollatorWithPadding. Supports ds["labels"] column access so the rest of
    the notebook (class weights, test labels) works unchanged.
    """
    def __init__(self, texts, labels, tokenizer, max_len, chunk=1000):
        ids, masks = [], []
        texts = list(texts)
        for i in range(0, len(texts), chunk):
            b = tokenizer(texts[i:i + chunk], truncation=True, max_length=max_len)
            ids.extend(b["input_ids"]); masks.extend(b["attention_mask"])
        self.input_ids = ids
        self.attention_mask = masks
        self.labels = [int(x) for x in labels]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        if isinstance(idx, str):                      # ds["labels"] -> whole column
            return getattr(self, "labels" if idx in ("label", "labels") else idx)
        return {"input_ids": self.input_ids[idx],
                "attention_mask": self.attention_mask[idx],
                "labels": self.labels[idx]}

def encode(df, tok, model_name):
    """Model-aware encoding: applies the right text prep for the given model."""
    texts = _prep_texts(df["_text"].tolist(), model_name)
    return TokenizedDataset(texts, df["labels"].tolist(), tok, MAX_LEN)

# tokenizer used by the (BERTweet-only) grid search below
grid_tok = make_tokenizer(MODEL_NAME)
_sanity = encode(train_df.head(3), grid_tok, MODEL_NAME)
print("Tokeniser helpers ready. grid tokenizer:", MODEL_NAME,
      "| example token lengths:", [len(_sanity[i]["input_ids"]) for i in range(3)])

Tokeniser helpers ready. grid tokenizer: vinai/bertweet-base | example token lengths: [13, 22, 22]


## 5. Weighted trainer + training routine

`WeightedTrainer` injects the inverse-frequency class weights into the loss so Neutral is upweighted. `TrainingArguments` is built in a version-robust way (the `evaluation_strategy`/`eval_strategy` rename differs across `transformers` versions).

In [9]:
import inspect, torch, torch.nn as nn
from transformers import (AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, EarlyStoppingCallback)
_TA_PARAMS = set(inspect.signature(TrainingArguments.__init__).parameters)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._cw = (torch.tensor(class_weight, dtype=torch.float)
                    if class_weight is not None else None)
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        w = self._cw.to(logits.device) if self._cw is not None else None
        loss = nn.CrossEntropyLoss(weight=w)(logits, labels)
        return (loss, outputs) if return_outputs else loss

def make_args(output_dir, lr, epochs, do_eval=True):
    common = dict(
        output_dir=str(output_dir), learning_rate=lr, num_train_epochs=epochs,
        per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
        weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO, seed=SEED,
        fp16=FP16, group_by_length=GROUP_BY_LENGTH, logging_steps=100,
        report_to="none", save_total_limit=1,
    )
    eval_key = ("eval_strategy" if "eval_strategy" in _TA_PARAMS else
                "evaluation_strategy" if "evaluation_strategy" in _TA_PARAMS else None)
    if eval_key:
        common[eval_key] = "epoch" if do_eval else "no"
    if do_eval and EARLY_STOP and eval_key:
        common.update(save_strategy="epoch", load_best_model_at_end=True,
                      metric_for_best_model="f1_macro", greater_is_better=True)
    dropped = [k for k in common if k not in _TA_PARAMS]
    common = {k: v for k, v in common.items() if k in _TA_PARAMS}
    if dropped:
        print("note: TrainingArguments version doesn't support", dropped, "- skipped")
    return TrainingArguments(**common)

def maybe_freeze(model, n):
    if not n:
        return
    base = model.base_model
    for p in base.embeddings.parameters():
        p.requires_grad = False
    for layer in base.encoder.layer[:n]:
        for p in layer.parameters():
            p.requires_grad = False
    print(f"Froze embeddings + first {n} encoder layers.")

def new_model(model_name):
    m = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=NUM_LABELS)
    maybe_freeze(m, FREEZE_BOTTOM)
    return m

def train_eval(train_ds, eval_ds, lr, epochs, weighted, model_name, tokenizer, tag="run"):
    """Fine-tune `model_name` on train_ds. Collator is built from the matching
    tokenizer so each architecture pads with its own pad token."""
    cw = class_weights(train_ds["labels"]) if weighted else None
    args = make_args(OUT_DIR / tag, lr, epochs, do_eval=eval_ds is not None)
    cbs = ([EarlyStoppingCallback(EARLY_STOP)]
           if (eval_ds is not None and EARLY_STOP) else None)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    trainer = WeightedTrainer(
        model=new_model(model_name), args=args, class_weight=cw,
        train_dataset=train_ds, eval_dataset=eval_ds,
        data_collator=collator, compute_metrics=compute_metrics, callbacks=cbs)
    trainer.train()
    metrics = trainer.evaluate() if eval_ds is not None else {}
    return trainer, metrics
print("Trainer ready (model-aware).")

Trainer ready (model-aware).


## 6. Grid search (validation-based; optional K-fold)

Searches `learning-rate × class-weighting` on a stratified **subsample** for speed, selecting on val macro-F1. Set `N_FOLDS > 1` for real StratifiedKFold (k× slower).

In [10]:
import itertools
from sklearn.model_selection import train_test_split, StratifiedKFold

# stratified subsample of the training set to keep search affordable
if GRID_SUBSAMPLE and GRID_SUBSAMPLE < len(train_df):
    sub_df, _ = train_test_split(train_df, train_size=GRID_SUBSAMPLE,
                                 stratify=train_df["labels"], random_state=SEED)
else:
    sub_df = train_df
print(f"Grid-search training pool: {len(sub_df)} rows (BERTweet only)")

def _enc(df):                      # grid encodes with the BERTweet tokenizer
    return encode(df, grid_tok, MODEL_NAME)

results = []
for lr, weighted in itertools.product(GRID_LR, GRID_WEIGHTED):
    tag = f"grid_lr{lr}_w{int(weighted)}"
    if N_FOLDS > 1:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
        fold_f1 = []
        X = sub_df.reset_index(drop=True)
        for k, (tr, va) in enumerate(skf.split(X, X["labels"])):
            _, m = train_eval(_enc(X.iloc[tr]), _enc(X.iloc[va]), lr, GRID_EPOCHS,
                              weighted, MODEL_NAME, grid_tok, tag=f"{tag}_f{k}")
            fold_f1.append(m["eval_f1_macro"])
        f1 = float(np.mean(fold_f1))
    else:
        tr_df, va_df = train_test_split(sub_df, test_size=0.15,
                                        stratify=sub_df["labels"], random_state=SEED)
        _, m = train_eval(_enc(tr_df), _enc(va_df), lr, GRID_EPOCHS,
                          weighted, MODEL_NAME, grid_tok, tag=tag)
        f1 = m["eval_f1_macro"]
    results.append({"lr": lr, "weighted": weighted, "val_f1_macro": round(f1, 4)})
    print(f"  {tag}: val macro-F1 = {f1:.4f}")

res_df = pd.DataFrame(results).sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
best = res_df.iloc[0]
print("\nGrid results:\n", res_df.to_string(index=False))
print(f"\nBest config -> lr={best.lr}, weighted={bool(best.weighted)}, "
      f"val macro-F1={best.val_f1_macro}")

Grid-search training pool: 20000 rows (BERTweet only)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


note: TrainingArguments version doesn't support ['group_by_length'] - skipped


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,0.616003,0.582072,0.693974,0.745000
2,0.461801,0.588782,0.735985,0.762333


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1 Macro,Accuracy
0.461801,0.588782,2,0.735985,0.762333


  grid_lr3e-05_w0: val macro-F1 = 0.7360

Grid results:
      lr  weighted  val_f1_macro
0.00002      True        0.7399
0.00001      True        0.7364
0.00003     False        0.7360
0.00003      True        0.7355
0.00002     False        0.7337
0.00001     False        0.7250

Best config -> lr=2e-05, weighted=True, val macro-F1=0.7399


## 7. Train both ensemble members on full train → collect val/test logits

The winning `(lr, weighted)` from the grid is reused to fine-tune **both** models on the full training split (val drives early stopping). For each model we store raw **validation** and **test** logits — the validation logits are what the threshold tuner fits on; the test logits are scored once at the end.

In [11]:
import numpy as np, gc

BEST_LR       = float(best.lr)
BEST_WEIGHTED = bool(best.weighted)
print(f"Final config: lr={BEST_LR}, weighted={BEST_WEIGHTED}, epochs={EPOCHS}")
print(f"Ensemble members: {MODELS}\n")

val_true  = np.array(val_df["labels"].tolist())
test_true = np.array(test_df["labels"].tolist())

val_logits_per_model  = []
test_logits_per_model = []
single_f1 = {}

for model_name in MODELS:
    print("=" * 72)
    print("Fine-tuning:", model_name)
    print("=" * 72)
    tok   = make_tokenizer(model_name)
    tr_ds = encode(train_df, tok, model_name)
    va_ds = encode(val_df,   tok, model_name)
    te_ds = encode(test_df,  tok, model_name)

    short = model_name.split("/")[-1]
    trainer, _ = train_eval(tr_ds, va_ds, BEST_LR, EPOCHS, BEST_WEIGHTED,
                            model_name, tok, tag=f"final_{short}")

    v_logits = np.asarray(trainer.predict(va_ds).predictions, dtype=np.float64)
    t_logits = np.asarray(trainer.predict(te_ds).predictions, dtype=np.float64)
    val_logits_per_model.append(v_logits)
    test_logits_per_model.append(t_logits)

    f1_s = f1_score(test_true, np.argmax(t_logits, 1), average="macro")
    single_f1[model_name] = f1_s
    print(f"\n[{short}] single-model TEST macro-F1 (argmax) = {f1_s:.4f}")

    # free GPU memory before the next member
    del trainer, tr_ds, va_ds, te_ds
    gc.collect()
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

print("\nCollected val/test logits for", len(MODELS), "models.")
print("Single-model TEST macro-F1:", {k.split("/")[-1]: round(v, 4) for k, v in single_f1.items()})

Final config: lr=2e-05, weighted=True, epochs=5
Ensemble members: ['vinai/bertweet-base', 'cardiffnlp/twitter-roberta-base-sentiment-latest']

Fine-tuning: vinai/bertweet-base


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


note: TrainingArguments version doesn't support ['group_by_length'] - skipped


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1 Macro,Accuracy
0.314757,0.541390,4,0.758178,0.778846



[twitter-roberta-base-sentiment-latest] single-model TEST macro-F1 (argmax) = 0.7544

Collected val/test logits for 2 models.
Single-model TEST macro-F1: {'bertweet-base': 0.7517, 'twitter-roberta-base-sentiment-latest': 0.7544}


## 8. Log-average ensemble + per-class threshold tuning → final TEST score

`log_softmax` each model, average across models (= geometric mean of probabilities), then fit per-class additive offsets on the **validation** ensemble to maximise macro-F1 and apply them to **test**. Both the plain argmax ensemble and the threshold-tuned ensemble are reported so the gain is visible, alongside the per-class breakdown (Neutral is the one to watch) and per-domain F1.

In [12]:
import numpy as np
from sklearn.metrics import f1_score, classification_report

def log_softmax_np(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - z.max(axis=1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=1, keepdims=True))

def tune_thresholds(val_scores, val_labels, grid=None, passes=4):
    """Coordinate-ascent per-class additive offsets maximising macro-F1.

    Low-capacity (one scalar per class) and FIT ON VALIDATION ONLY. The same
    offsets are applied to test unchanged, so there is no test leakage. Argmax
    of (scores + offsets) shifts the decision boundary toward under-predicted
    classes (here: Neutral)."""
    if grid is None:
        grid = np.linspace(-3.0, 3.0, 61)
    n_cls = val_scores.shape[1]
    offs  = np.zeros(n_cls)
    def f1_at(o):
        return f1_score(val_labels, np.argmax(val_scores + o, 1), average="macro")
    best = f1_at(offs)
    for _ in range(passes):
        improved = False
        for c in range(n_cls):
            best_g = offs[c]
            for g in grid:
                trial = offs.copy(); trial[c] = g
                s = f1_at(trial)
                if s > best + 1e-9:
                    best, best_g, improved = s, g, True
            offs[c] = best_g
        if not improved:
            break
    return offs, best

LABELS = ["Negative(0)", "Neutral(1)", "Positive(2)"]

def report(true, pred, header):
    f1 = f1_score(true, pred, average="macro")
    print(f"\n=== {header}: macro-F1 = {f1:.4f} ({f1*100:.1f}%) ===")
    print(classification_report(true, pred, target_names=LABELS, digits=3))
    return f1

# --- log-averaged ensemble ---
val_lp   = [log_softmax_np(v) for v in val_logits_per_model]
test_lp  = [log_softmax_np(t) for t in test_logits_per_model]
val_ens  = np.mean(val_lp,  axis=0)
test_ens = np.mean(test_lp, axis=0)

# 1) ensemble, plain argmax (no tuning)
f1_argmax = report(test_true, np.argmax(test_ens, 1),
                   "ENSEMBLE (log-avg) + argmax")

# 2) ensemble + per-class thresholds fit on VAL, applied to TEST
offsets, val_best = tune_thresholds(val_ens, val_true, grid=THRESH_GRID)
print(f"\nTuned per-class offsets (fit on val): {np.round(offsets, 3)}"
      f"  | val macro-F1 after tuning = {val_best:.4f}")
test_pred = np.argmax(test_ens + offsets, 1)
f1_tuned  = report(test_true, test_pred,
                   "ENSEMBLE (log-avg) + per-class threshold tuning  [FINAL]")

# --- summary table ---
print("\n" + "-" * 56)
print("SUMMARY (TEST macro-F1)")
print("-" * 56)
for k, v in single_f1.items():
    print(f"  {k.split('/')[-1]:<38} {v:.4f}")
print(f"  {'ensemble (argmax)':<38} {f1_argmax:.4f}")
print(f"  {'ensemble + thresholds  [FINAL]':<38} {f1_tuned:.4f}  (Δ {f1_tuned - f1_argmax:+.4f})")
print(f"  {'Route A baseline':<38} {ROUTE_A_F1:.2f}")

# --- per-domain macro-F1 on the FINAL (tuned) predictions ---
_type_col = _find_col(test_df.columns, ["type", "source", "domain"])
if _type_col is not None:
    print("\nPer-domain macro-F1 (final tuned predictions):")
    _t = test_df[_type_col].astype(str).str.lower().to_numpy()
    for _dom in pd.unique(_t):
        _m = _t == _dom
        print(f"  {_dom:>7}: {f1_score(test_true[_m], test_pred[_m], average='macro'):.4f}"
              f"   (n={_m.sum():,})")
else:
    print("\n(no Type/source column in split — per-domain F1 skipped)")


=== ENSEMBLE (log-avg) + argmax: macro-F1 = 0.7569 (75.7%) ===
              precision    recall  f1-score   support

 Negative(0)      0.806     0.826     0.816     13468
  Neutral(1)      0.582     0.602     0.592      9528
 Positive(2)      0.883     0.845     0.863     15267

    accuracy                          0.778     38263
   macro avg      0.757     0.757     0.757     38263
weighted avg      0.781     0.778     0.779     38263


Tuned per-class offsets (fit on val): [0.1 0.2 0. ]  | val macro-F1 after tuning = 0.7617

=== ENSEMBLE (log-avg) + per-class threshold tuning  [FINAL]: macro-F1 = 0.7580 (75.8%) ===
              precision    recall  f1-score   support

 Negative(0)      0.815     0.811     0.813     13468
  Neutral(1)      0.570     0.632     0.599      9528
 Positive(2)      0.891     0.834     0.862     15267

    accuracy                          0.776     38263
   macro avg      0.759     0.759     0.758     38263
weighted avg      0.784     0.776     0.779  

## Notes

- **Threshold tuning is leakage-safe:** offsets are fit only on the validation ensemble, then applied to test unchanged. If `val macro-F1 after tuning` jumps a lot but test does not, the val set is too small for stable offsets — widen the search or fall back to argmax.
- **If twitter-roberta underperforms BERTweet**, the log-average can still help (diverse errors), but check the single-model lines in the summary; if roberta is far behind, consider a weighted average favouring BERTweet.
- **Deployment:** the case-manual `.model` pickle (`sentiment_deploy.py`) wraps a *single* BERTweet model. The ensemble + offsets are an evaluation/leaderboard artifact; packaging both models and the offset vector into one `{"vectorizer", "classifier"}` deployment is a separate step if the API must serve the ensemble.